# 千际轻量四源数据基座：本地演示

这个 Notebook 先用模拟数据跑通以下闭环：

`数据源 → 字段标准化 → SQLite → Python/REST/Excel → OpenBB Provider`

模拟流程成功后，只需要填写商业数据源凭据并把 `SOURCE_TO_RUN` 改成 `tushare`、`ifind`、`wind` 或 `choice`。不会把密码或 Token 写入代码。

## 1. 安装项目

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidates if (path / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请先解压项目，并在 VS Code 中打开整个 qianji_openbb_mini 文件夹。")
os.chdir(PROJECT_ROOT)
print("项目目录：", PROJECT_ROOT.resolve())

In [ ]:
%pip install -q -e ".[dev]"

安装完成后，如果 VS Code 提示重启内核，请点击重启，再从下一格继续。

## 2. 创建本地配置和数据库

In [ ]:
import shutil

env_file = PROJECT_ROOT / ".env"
if not env_file.exists():
    shutil.copyfile(PROJECT_ROOT / ".env.example", env_file)
    print("已创建 .env。真实接入前，用 VS Code 打开它并填写对应 Token。")
else:
    print(".env 已存在，不覆盖。")

from dotenv import load_dotenv
load_dotenv(env_file, override=True)

from qianji_data_mini import Database

database = Database()
print("SQLite 数据库：", database.path)

## 3. 用模拟数据完成下载和落库

In [ ]:
from datetime import date, timedelta
from qianji_data_mini import ingest_daily

end_date = date.today()
start_date = end_date - timedelta(days=30)

result = ingest_daily(
    source="mock",
    symbols=["000001.SZ", "600000.SH"],
    start_date=start_date,
    end_date=end_date,
    database_path=database.path,
)
result

In [ ]:
frame = database.query_dataframe(
    symbol="000001.SZ",
    start_date=start_date,
    end_date=end_date,
    source="auto",
)
print(f"查询到 {len(frame)} 条")
display(frame.tail(10))
display(database.source_status())

## 4. 模拟客户端调用 REST 接口（不需要手工启动服务器）

In [ ]:
from fastapi.testclient import TestClient
from qianji_data_mini.service import create_app

client = TestClient(create_app(database.path))

health = client.get("/health")
response = client.get(
    "/v1/equity/price/historical",
    params={
        "symbol": "000001.SZ",
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "source": "auto",
    },
)

print("健康检查：", health.json())
print("行情接口 HTTP：", response.status_code)
payload = response.json()
display(__import__("pandas").DataFrame(payload["results"]).tail())

## 5. 导出给研究员使用的 Excel

In [ ]:
excel_path = PROJECT_ROOT / "千际本地行情演示.xlsx"
with __import__("pandas").ExcelWriter(excel_path, engine="openpyxl") as writer:
    frame.to_excel(writer, sheet_name="000001.SZ日线", index=False)
    database.source_status().to_excel(writer, sheet_name="数据源状态", index=False)
print("已导出：", excel_path.resolve())

## 6. 切换到真实数据源

先打开 `.env`：

- Tushare：填写 `TUSHARE_TOKEN`；
- iFinD：填写 `IFIND_REFRESH_TOKEN`；
- Wind：确保当前 Python 可以 `import WindPy`；
- Choice：确保当前 Python 可以 `import EmQuantAPI`，并完成官方激活。

然后把下面的 `SOURCE_TO_RUN` 改成对应名称。第一次只测试一个代码、较短日期，确认字段和权限后再扩大。

In [ ]:
# 只读检查：不会登录商业接口，也不会消耗额度。
exec((PROJECT_ROOT / "clients" / "check_environment.py").read_text(encoding="utf-8"))

In [ ]:
SOURCE_TO_RUN = "mock"  # 可改为 tushare / ifind / wind / choice
REAL_SYMBOLS = ["000001.SZ"]

real_result = ingest_daily(
    source=SOURCE_TO_RUN,
    symbols=REAL_SYMBOLS,
    start_date=start_date,
    end_date=end_date,
    database_path=database.path,
)
real_result

## 7. 安装并注册 OpenBB `qianji` Provider（可选）

In [ ]:
# 需要验证 OpenBB 时，删除下面两行开头的 #，运行本格并等待安装完成。
# %pip install -q -e ".[openbb]"
# !openbb-build
print("完成安装和 openbb-build 后，必须重启 Jupyter 内核，再运行下一格。")

In [ ]:
# 仅在上一格安装 OpenBB、执行 openbb-build 并重启内核后运行。
try:
    from openbb import obb
    openbb_result = obb.equity.price.historical(
        symbol="000001.SZ",
        start_date=start_date.isoformat(),
        end_date=end_date.isoformat(),
        provider="qianji",
        source="auto",
    )
    display(openbb_result.to_dataframe().tail())
except ImportError:
    print("尚未安装 OpenBB，可先完成前面的 SQLite、REST 和 Excel 验证。")
except Exception as exc:
    print("OpenBB 调用未完成：", type(exc).__name__, exc)
    print("确认已运行 openbb-build 并重启内核。")

## 本 Notebook 能证明什么

- 模拟源能够下载、标准化、幂等写入 SQLite；
- Python、REST、Excel 三种消费方式可用；
- OpenBB 私有 Provider 已准备好从公司库读取。

真实四源能否成功，最终取决于本机 SDK、账号购买权限、接口流量、IP 白名单和合同允许的存储范围。